# 01 - Camada Bronze: ingestão bruta em Delta
Pipeline de limpeza em etapas, aplicado sobre a camada Bronze:

In [0]:
from pyspark.sql import functions as F

arquivos = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

for tabela, arquivo in arquivos.items():
    caminho = f"/Volumes/olist_project/bronze/raw_files/{arquivo}"
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False)
          .csv(caminho)
          .withColumn("_ingest_timestamp", F.current_timestamp())
          .withColumn("_source_file", F.lit(arquivo)))

    df.write.format("delta").mode("overwrite") \
        .saveAsTable(f"olist_project.bronze.{tabela}")

    print(f"✅ bronze.{tabela}: {df.count()} linhas")

Corrigindo arquivo reviews

In [0]:
from pyspark.sql import functions as F

df_reviews = (spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("multiLine", True)      # permite campos de texto com quebras de linha
    .option("quote", '"')            # trata texto entre aspas como um único campo
    .option("escape", '"')           # trata aspas duplas escapadas dentro do texto
    .csv("/Volumes/olist_project/bronze/raw_files/olist_order_reviews_dataset.csv")
    .withColumn("_ingest_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.lit("olist_order_reviews_dataset.csv"))
)

df_reviews.write.format("delta").mode("overwrite") \
    .saveAsTable("olist_project.bronze.reviews")

print("✅ bronze.reviews recriada:", df_reviews.count(), "linhas")

In [0]:
#display(spark.table("olist_project.bronze.reviews").limit(10))